# Syndrome Cycles

In this notebook, we implement a full **quantum error correction (QEC)** cycle using the 3-qubit **bit-flip code**.

We will:
1. **Encode** a logical qubit  
2. **Apply** a single-qubit error  
3. **Measure** the syndrome using stabilizers  
4. **Recover** the correct state  
5. **Decode** back to the logical qubit and compare

### ✅ Requirements
- Python 3.8+
- `numpy`

In [ ]:
import numpy as np
import random

I = np.eye(2)
X = np.array([[0, 1], [1, 0]])
Z = np.array([[1, 0], [0, -1]])

ket0 = np.array([1, 0])
ket1 = np.array([0, 1])

def tensor(*ops):
    result = ops[0]
    for op in ops[1:]:
        result = np.kron(result, op)
    return result

### Step 1: Encode Logical Qubit
We encode $|\psi\rangle = \alpha|000\rangle + \beta|111\rangle$.

In [ ]:
alpha = 1 / np.sqrt(2)
beta = 1 / np.sqrt(2)

psi0 = tensor(ket0, ket0, ket0)
psi1 = tensor(ket1, ket1, ket1)

encoded_state = alpha * psi0 + beta * psi1
encoded_state

### Step 2: Apply Random X Error
Flip a single qubit randomly to simulate a bit-flip error.

In [ ]:
qubit = random.choice([0, 1, 2])
ops = [I, I, I]
ops[qubit] = X

error_op = tensor(*ops)
noisy_state = error_op @ encoded_state
print(f"Error applied to qubit {qubit}")

### Step 3: Measure Syndrome
We use stabilizers $Z_1Z_2$ and $Z_2Z_3$ to detect error syndrome.

In [ ]:
Z1Z2 = tensor(Z, Z, I)
Z2Z3 = tensor(I, Z, Z)

s1 = np.allclose(Z1Z2 @ noisy_state, noisy_state)
s2 = np.allclose(Z2Z3 @ noisy_state, noisy_state)

syndrome = (int(not s1), int(not s2))
print("Syndrome:", syndrome)

### Step 4: Recover
We use majority logic: flip the qubit that disagrees.

In [ ]:
if syndrome == (1, 0):
    fix_index = 0
elif syndrome == (1, 1):
    fix_index = 1
elif syndrome == (0, 1):
    fix_index = 2
else:
    fix_index = None

if fix_index is not None:
    ops = [I, I, I]
    ops[fix_index] = X
    recovery_op = tensor(*ops)
    recovered_state = recovery_op @ noisy_state
    print(f"Applied recovery on qubit {fix_index}")
else:
    recovered_state = noisy_state
    print("No correction needed")

### Step 5: Decode
Project back to logical space.

In [ ]:
amp0 = np.vdot(psi0, recovered_state)
amp1 = np.vdot(psi1, recovered_state)
norm = np.sqrt(abs(amp0)**2 + abs(amp1)**2)

logical = np.array([amp0, amp1]) / norm
print("Decoded logical qubit:", logical)